# Bagging — IEEE-CIS Fraud Detection

MLflow experiment: `Bagging_Training`

Bagging — bootstrap ensemble. Variance-ს ამცირებს.

## Setup

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")

# Kaggle-ზე uncomment-ი:
# from kaggle_secrets import UserSecretsClient
# s = UserSecretsClient()
# os.environ["DAGSHUB_TOKEN"]    = s.get_secret("DAGSHUB_TOKEN")
# os.environ["DAGSHUB_USERNAME"] = s.get_secret("DAGSHUB_USERNAME")
# os.environ["DAGSHUB_REPO"]     = s.get_secret("DAGSHUB_REPO")

import dagshub, mlflow
dagshub.init(
    repo_owner=os.environ["DAGSHUB_USERNAME"],
    repo_name=os.environ["DAGSHUB_REPO"],
    mlflow=True,
)
os.environ["MLFLOW_TRACKING_USERNAME"] = os.environ["DAGSHUB_USERNAME"]
os.environ["MLFLOW_TRACKING_PASSWORD"] = os.environ["DAGSHUB_TOKEN"]

mlflow.set_experiment("Bagging_Training")

## Data Loading

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

KAGGLE = "/kaggle/input/ieee-fraud-detection"
LOCAL  = "./data"
PATH   = KAGGLE if os.path.exists(KAGGLE) else LOCAL

tx = pd.read_csv(f"{PATH}/train_transaction.csv")
id_ = pd.read_csv(f"{PATH}/train_identity.csv")
df = tx.merge(id_, on="TransactionID", how="left")
del tx, id_

print("shape:", df.shape)
print("fraud rate:", round(df["isFraud"].mean(), 4))


## Cleaning

In [ ]:
with mlflow.start_run(run_name="Bagging_Cleaning"):
    null_rate = df.isna().mean()
    drop_cols = null_rate[null_rate > 0.95].index.tolist()
    df.drop(columns=drop_cols, inplace=True)
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)

    mlflow.log_params({"null_threshold": 0.95, "cols_dropped": len(drop_cols)})
    mlflow.log_metrics({"rows": len(df), "fraud_rate": float(df["isFraud"].mean()), "cols": df.shape[1]})
    print("Cleaning:", df.shape)


## Feature Engineering

In [ ]:
with mlflow.start_run(run_name="Bagging_Feature_Engineering"):
    df["TransactionAmt_log"] = np.log1p(df["TransactionAmt"])

    d_cols = [c for c in df.columns if c.startswith("D") and c[1:].isdigit()]
    c_cols = [c for c in df.columns if c.startswith("C") and c[1:].isdigit()]
    m_cols = [c for c in df.columns if c.startswith("M") and c[1:].isdigit()]

    if d_cols:
        df["D_mean"]       = df[d_cols].mean(axis=1)
        df["D_null_count"] = df[d_cols].isna().sum(axis=1)
    if c_cols:
        df["C_sum"]        = df[c_cols].fillna(0).sum(axis=1)
    if m_cols:
        df["M_T_count"]    = (df[m_cols] == "T").sum(axis=1)
        df["M_F_count"]    = (df[m_cols] == "F").sum(axis=1)

    mlflow.log_metrics({"cols_after_fe": df.shape[1]})
    print("FE done:", df.shape)


## Feature Selection

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.impute import SimpleImputer

y = df["isFraud"]
X = df.drop(columns=["isFraud", "TransactionID"], errors="ignore")
num_cols = X.select_dtypes(include=np.number).columns.tolist()
cat_cols = X.select_dtypes(include="object").columns.tolist()

with mlflow.start_run(run_name="Bagging_Feature_Selection"):
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
        ("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))
        ]), cat_cols),
    ])
    sel = Pipeline([("prep", pre),
                    ("clf", ExtraTreesClassifier(n_estimators=50, random_state=42, n_jobs=-1))])
    sel.fit(X, y)
    imp = sel.named_steps["clf"].feature_importances_
    names = num_cols + cat_cols
    idx = np.argsort(imp)[::-1]
    cumulative = np.cumsum(imp[idx])
    k = int(np.searchsorted(cumulative, 0.97)) + 1
    selected = [names[i] for i in idx[:k] if i < len(names)]

    X = X[selected]
    num_cols = [c for c in num_cols if c in selected]
    cat_cols = [c for c in cat_cols if c in selected]

    mlflow.log_params({"method": "ExtraTrees_97pct", "n_features": len(selected)})
    mlflow.log_metrics({"features_kept": len(selected)})
    print("Selected:", len(selected), "features")


## Training

### Underfit  ძალიან ცოტა

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from sklearn.metrics import confusion_matrix, classification_report
from mlflow.models.signature import infer_signature
import tempfile, pathlib
from sklearn.ensemble import BaggingClassifier

num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())])
cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("enc", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])
prep = ColumnTransformer([("num", num_pipe, num_cols), ("cat", cat_pipe, cat_cols)])

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

with mlflow.start_run(run_name="Bagging_Underfit"):
    m = BaggingClassifier(n_estimators=5, max_samples=0.3, random_state=42, n_jobs=-1)
    p = Pipeline([("prep", prep), ("clf", m)])
    p.fit(X_tr, y_tr)
    tr = roc_auc_score(y_tr, p.predict_proba(X_tr)[:,1])
    te = roc_auc_score(y_te, p.predict_proba(X_te)[:,1])
    mlflow.log_params({**{'n_estimators': 5, 'max_samples': 0.3}, "note": "underfit"})
    mlflow.log_metrics({"train_auc": tr, "test_auc": te, "gap": tr - te})
    print(f"Underfit  train={tr:.4f} test={te:.4f} gap={tr-te:.4f}")


### Overfit  სრული სამი

In [ ]:
with mlflow.start_run(run_name="Bagging_Overfit"):
    m2 = BaggingClassifier(n_estimators=200, max_samples=1.0, max_features=1.0, bootstrap=True, random_state=42, n_jobs=-1)
    p2 = Pipeline([("prep", prep), ("clf", m2)])
    p2.fit(X_tr, y_tr)
    tr2 = roc_auc_score(y_tr, p2.predict_proba(X_tr)[:,1])
    te2 = roc_auc_score(y_te, p2.predict_proba(X_te)[:,1])
    mlflow.log_params({**{'n_estimators': 200, 'max_samples': 1.0}, "note": "overfit"})
    mlflow.log_metrics({"train_auc": tr2, "test_auc": te2, "gap": tr2 - te2})
    print(f"Overfit  train={tr2:.4f} test={te2:.4f} gap={tr2-te2:.4f}")


### Best Model

In [ ]:
with mlflow.start_run(run_name="Bagging_BestModel") as run:
    best = BaggingClassifier(n_estimators=50, max_samples=0.8, max_features=0.8, bootstrap=True, random_state=42, n_jobs=-1)
    bp = Pipeline([("prep", prep), ("clf", best)])
    bp.fit(X_tr, y_tr)

    y_pred  = bp.predict(X_te)
    y_score = bp.predict_proba(X_te)[:,1]

    mlflow.log_params({'n_estimators': 50, 'max_samples': 0.8, 'max_features': 0.8})
    mlflow.log_metrics({
        "test_auc":  float(roc_auc_score(y_te, y_score)),
        "train_auc": float(roc_auc_score(y_tr, bp.predict_proba(X_tr)[:,1])),
        "f1":        float(f1_score(y_te, y_pred, zero_division=0)),
        "precision": float(precision_score(y_te, y_pred, zero_division=0)),
        "recall":    float(recall_score(y_te, y_pred, zero_division=0)),
    })

    with tempfile.TemporaryDirectory() as tmp:
        tmp = pathlib.Path(tmp)
        cm = confusion_matrix(y_te, y_pred)
        fig, ax = plt.subplots(figsize=(4,3))
        import seaborn as sns
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                    xticklabels=["Legit","Fraud"], yticklabels=["Legit","Fraud"])
        fig.tight_layout(); fig.savefig(tmp/"confusion_matrix.png", dpi=100); plt.close()
        (tmp/"report.txt").write_text(
            classification_report(y_te, y_pred, target_names=["Legit","Fraud"], zero_division=0))
        mlflow.log_artifacts(str(tmp))

    sig = infer_signature(X_tr, bp.predict(X_tr))
    mlflow.sklearn.log_model(bp, artifact_path="model", signature=sig)
    print("Saved. run_id:", run.info.run_id)


## Cross-Validation

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_validate

with mlflow.start_run(run_name="Bagging_CrossValidation"):
    m_cv = BaggingClassifier(n_estimators=50, max_samples=0.8, max_features=0.8, bootstrap=True, random_state=42, n_jobs=-1)
    p_cv = Pipeline([("prep", prep), ("clf", m_cv)])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    res = cross_validate(p_cv, X, y, cv=cv,
                         scoring=["roc_auc", "f1", "precision", "recall"],
                         return_train_score=True, n_jobs=-1)

    mlflow.log_metrics({
        "cv_test_auc_mean":  float(res["test_roc_auc"].mean()),
        "cv_test_auc_std":   float(res["test_roc_auc"].std()),
        "cv_train_auc_mean": float(res["train_roc_auc"].mean()),
        "cv_test_f1_mean":   float(res["test_f1"].mean()),
    })
    print(f"CV AUC: {res['test_roc_auc'].mean():.4f} ± {res['test_roc_auc'].std():.4f}")
